# Daily Challenge – Superstore Sales Analysis
**Dataset :** Sample - Superstore  
**Concepts :** Diagnostic analysis, time-series, geographic analysis, discount strategy, interactive dashboards  
**Outils :** Pandas, NumPy, Matplotlib, Seaborn, ipywidgets

## Partie 1 – Chargement et préparation des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact, Dropdown, IntSlider
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
sns.set_theme(style='whitegrid')

In [ ]:
df = pd.read_csv('Sample - Superstore.csv', encoding='latin1')

print('Dimensions :', df.shape)
print('\nColonnes :')
print(df.columns.tolist())
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Vérifier les valeurs manquantes et les doublons
print('Valeurs manquantes par colonne :')
print(df.isnull().sum())

print(f'\nLignes dupliquées : {df.duplicated().sum()}')
df = df.drop_duplicates()

# Remplir les codes postaux manquants par 0
if 'Postal Code' in df.columns:
    df['Postal Code'] = df['Postal Code'].fillna(0)

In [ ]:
# Convertir les colonnes de dates en datetime
date_columns = ['Order Date', 'Ship Date']
for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

print('Types après conversion :')
print(df[date_columns].dtypes)

In [ ]:
# Feature engineering : nouvelles colonnes utiles
df['Profit Margin']    = (df['Profit'] / df['Sales']) * 100
df['Order Year']       = df['Order Date'].dt.year
df['Order Month']      = df['Order Date'].dt.month
df['Order Month-Year'] = df['Order Date'].dt.to_period('M')

print('Nouvelles colonnes créées :')
df[['Sales', 'Profit', 'Profit Margin', 'Order Year', 'Order Month']].head()

## Partie 2 – Analyse exploratoire avec Matplotlib

### 2.1 Tendance mensuelle des ventes (interactive)

In [ ]:
# Préparer les données de ventes mensuelles par catégorie
monthly_sales = df.groupby(['Order Month-Year', 'Category'])['Sales'].sum().reset_index()
monthly_sales['Date'] = monthly_sales['Order Month-Year'].dt.to_timestamp()

def plot_monthly_sales(category='All'):
    plt.figure(figsize=(12, 6))

    if category == 'All':
        total_monthly = df.groupby('Order Month-Year')['Sales'].sum()
        plt.plot(total_monthly.index.to_timestamp(), total_monthly.values,
                 marker='o', linewidth=2, markersize=4, color='steelblue')
        plt.title('Monthly Sales Trend – All Categories', fontsize=14, fontweight='bold')
    else:
        cat_data = monthly_sales[monthly_sales['Category'] == category]
        plt.plot(cat_data['Date'], cat_data['Sales'],
                 marker='o', linewidth=2, markersize=4, color='coral')
        plt.title(f'Monthly Sales Trend – {category}', fontsize=14, fontweight='bold')

    plt.xlabel('Date')
    plt.ylabel('Sales ($)')
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Widget interactif
categories = ['All'] + list(df['Category'].unique())
category_dropdown = Dropdown(options=categories, value='All', description='Category:')
interact(plot_monthly_sales, category=category_dropdown);

### 2.2 Performance géographique des ventes (interactive)

In [ ]:
# Ventes totales par état, triées par montant
state_sales = df.groupby('State')['Sales'].sum().sort_values(ascending=True)

def plot_top_states(top_n=10):
    top_states = state_sales.tail(top_n)

    plt.figure(figsize=(12, max(6, top_n * 0.4)))
    plt.barh(range(len(top_states)), top_states.values, color='steelblue')
    plt.yticks(range(len(top_states)), top_states.index)
    plt.xlabel('Total Sales ($)')
    plt.ylabel('State')
    plt.title(f'Top {top_n} States by Sales', fontsize=14, fontweight='bold')

    # Afficher les valeurs sur les barres
    max_val = top_states.values.max()
    for i, (state, value) in enumerate(top_states.items()):
        plt.text(value + max_val * 0.01, i, f'${value:,.0f}', va='center', fontsize=9)

    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f'Top {top_n} états → ${top_states.sum():,.0f} en ventes')

top_n_slider = IntSlider(min=5, max=25, value=10, description='Top N States:')
interact(plot_top_states, top_n=top_n_slider);

## Partie 3 – Communiquer les insights avec Seaborn

### 3.1 Top 10 produits les plus rentables

In [ ]:
# Top 10 produits par profit total
product_profit = df.groupby('Product Name')['Profit'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 8))
ax = sns.barplot(x=product_profit.values, y=product_profit.index, palette='viridis')

plt.title('Top 10 Most Profitable Products\nExecutive Summary – Product Performance', 
          fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Total Profit ($)', fontweight='bold')
plt.ylabel('Product Name', fontweight='bold')

# Annoter les barres avec les valeurs exactes
max_val = product_profit.values.max()
for i, (product, profit) in enumerate(product_profit.items()):
    ax.text(profit + max_val * 0.01, i, f'${profit:,.0f}', va='center', fontsize=9, fontweight='bold')

plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Produit le plus rentable  : {product_profit.index[0]}')
print(f'Profit du top 1           : ${product_profit.iloc[0]:,.0f}')
print(f'Profit total top 10       : ${product_profit.sum():,.0f}')
print(f'Profit moyen top 10       : ${product_profit.mean():,.0f}')

### 3.2 Discount vs Profit – Analyse de la stratégie de réduction

In [ ]:
plt.figure(figsize=(14, 7))

# Scatter plot coloré par catégorie
sns.scatterplot(data=df, x='Discount', y='Profit', hue='Category', alpha=0.6, s=50)

# Ligne de tendance globale
sns.regplot(data=df, x='Discount', y='Profit', scatter=False,
            color='red', line_kws={'linewidth': 2, 'linestyle': '--'})

# Ligne de seuil de rentabilité
plt.axhline(y=0, color='black', linestyle='-', alpha=0.4, linewidth=1)
plt.text(0.5, 20, 'Break-even (Profit = 0)', fontsize=9, alpha=0.6)

plt.title('Discount Strategy: Impact on Profitability by Category',
          fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Discount Rate', fontweight='bold')
plt.ylabel('Profit ($)', fontweight='bold')
plt.legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Analyse des remises élevées (> 20%)
high_discount = df[df['Discount'] > 0.2]
print(f'Transactions avec remise > 20%        : {len(high_discount):,}')
print(f'Profit moyen pour ces transactions    : ${high_discount["Profit"].mean():.2f}')
pct_loss = (high_discount['Profit'] < 0).mean() * 100
print(f'% de ventes à perte (discount > 20%)  : {pct_loss:.1f}%')

print('\nImpact par catégorie (discount > 20%) :')
for cat in df['Category'].unique():
    cat_data = df[(df['Category'] == cat) & (df['Discount'] > 0.2)]
    if len(cat_data) > 0:
        print(f'  {cat} : profit moyen = ${cat_data["Profit"].mean():.2f}')

## Partie 4 – Comparaison Matplotlib vs Seaborn

In [ ]:
import time

# Mesurer le temps d'un graphique Matplotlib simple
start = time.time()
plt.figure(figsize=(8, 5))
plt.plot(df.groupby('Order Year')['Sales'].sum())
plt.close()
matplotlib_time = time.time() - start

# Mesurer le temps de l'équivalent Seaborn
start = time.time()
plt.figure(figsize=(8, 5))
sns.lineplot(data=df.groupby('Order Year')['Sales'].sum().reset_index(),
             x='Order Year', y='Sales')
plt.close()
seaborn_time = time.time() - start

print('=== COMPARAISON MATPLOTLIB vs SEABORN ===')
print()
print('MATPLOTLIB – Points forts :')
print('  • Contrôle précis sur chaque élément du graphique')
print('  • Intégration native avec ipywidgets (interactivité)')
print('  • Annotations et positionnement de texte très flexibles')
print('  • Idéal pour les graphiques personnalisés et complexes')
print()
print('SEABORN – Points forts :')
print('  • Style visuel professionnel par défaut')
print('  • Visualisations statistiques intégrées (regplot, boxplot...)')
print('  • Gestion automatique des couleurs et légendes')
print('  • Idéal pour communiquer des résultats à des non-techniciens')
print()
print(f'Temps Matplotlib : {matplotlib_time:.4f} secondes')
print(f'Temps Seaborn    : {seaborn_time:.4f} secondes')

### Recommandation

**Pour l'exploration rapide**, je préfère **Matplotlib** : il offre un rendu plus rapide pour les graphiques simples et s'intègre parfaitement avec `ipywidgets` pour des analyses dynamiques et interactives.

**Pour les présentations aux parties prenantes**, je préfère **Seaborn** : il produit des graphiques prêts à publier, inclut des fonctionnalités statistiques intégrées (`regplot`, intervalles de confiance), et propose des palettes de couleurs professionnelles qui améliorent la communication des insights.

## Partie 5 – Dashboard interactif multi-graphiques

In [ ]:
def create_dashboard():
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

    # Graphique 1 : Tendance mensuelle des ventes
    monthly_total = df.groupby('Order Month-Year')['Sales'].sum()
    ax1.plot(monthly_total.index.to_timestamp(), monthly_total.values,
             marker='o', markersize=3, color='steelblue')
    ax1.set_title('Monthly Sales Trend', fontweight='bold')
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Sales ($)')
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(True, alpha=0.3)

    # Graphique 2 : Ventes par catégorie
    cat_sales = df.groupby('Category')['Sales'].sum()
    ax2.bar(cat_sales.index, cat_sales.values, color=['steelblue', 'coral', 'green'])
    ax2.set_title('Sales by Category', fontweight='bold')
    ax2.set_xlabel('Category')
    ax2.set_ylabel('Sales ($)')
    ax2.grid(axis='y', alpha=0.3)

    # Graphique 3 : Top 10 états par ventes
    top_10_states = state_sales.tail(10)
    ax3.barh(range(len(top_10_states)), top_10_states.values, color='steelblue')
    ax3.set_yticks(range(len(top_10_states)))
    ax3.set_yticklabels(top_10_states.index)
    ax3.set_title('Top 10 States by Sales', fontweight='bold')
    ax3.set_xlabel('Sales ($)')
    ax3.grid(axis='x', alpha=0.3)

    # Graphique 4 : Discount vs Profit
    colors = {'Furniture': 'steelblue', 'Office Supplies': 'coral', 'Technology': 'green'}
    for cat in df['Category'].unique():
        cat_data = df[df['Category'] == cat]
        ax4.scatter(cat_data['Discount'], cat_data['Profit'],
                    label=cat, alpha=0.5, s=20, color=colors.get(cat, 'gray'))
    ax4.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    ax4.set_title('Discount vs Profit by Category', fontweight='bold')
    ax4.set_xlabel('Discount')
    ax4.set_ylabel('Profit ($)')
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    plt.suptitle('Superstore Performance Dashboard', fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

create_dashboard()

## Bonus – Annotation des outliers (Discount vs Profit)

In [ ]:
plt.figure(figsize=(12, 7))
sns.scatterplot(data=df, x='Discount', y='Profit', hue='Category', alpha=0.5, s=40)

# Top 3 transactions les plus rentables
top_3 = df.nlargest(3, 'Profit')
for _, row in top_3.iterrows():
    plt.annotate(f'+${row["Profit"]:.0f}',
                 xy=(row['Discount'], row['Profit']),
                 xytext=(10, 5), textcoords='offset points',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='#2ecc71', alpha=0.7),
                 arrowprops=dict(arrowstyle='->', color='green'),
                 fontsize=8)

# Top 3 transactions les plus déficitaires
bottom_3 = df.nsmallest(3, 'Profit')
for _, row in bottom_3.iterrows():
    plt.annotate(f'${row["Profit"]:.0f}',
                 xy=(row['Discount'], row['Profit']),
                 xytext=(10, -15), textcoords='offset points',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='#e74c3c', alpha=0.7),
                 arrowprops=dict(arrowstyle='->', color='red'),
                 fontsize=8)

plt.axhline(y=0, color='black', linestyle='--', alpha=0.4)
plt.title('Discount vs Profit – Identification des outliers', fontsize=13, fontweight='bold')
plt.xlabel('Discount')
plt.ylabel('Profit ($)')
plt.legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Résumé Exécutif – Key Findings

In [ ]:
total_sales    = df['Sales'].sum()
total_profit   = df['Profit'].sum()
profit_margin  = (total_profit / total_sales) * 100

top_state      = state_sales.index[-1]
top_state_val  = state_sales.iloc[-1]
top5_pct       = (state_sales.tail(5).sum() / total_sales) * 100

top_category   = df.groupby('Category')['Sales'].sum().sort_values(ascending=False).index[0]
top_product    = product_profit.index[0]

loss_rate = (df[df['Discount'] > 0.2]['Profit'] < 0).mean() * 100

print('=== EXECUTIVE SUMMARY – KEY FINDINGS ===')
print()
print('PERFORMANCE GLOBALE :')
print(f'  Chiffre d\'affaires total : ${total_sales:,.0f}')
print(f'  Profit total             : ${total_profit:,.0f}')
print(f'  Marge bénéficiaire       : {profit_margin:.1f}%')
print()
print('PERFORMANCE GÉOGRAPHIQUE :')
print(f'  État le plus performant  : {top_state} (${top_state_val:,.0f})')
print(f'  Top 5 états              : {top5_pct:.1f}% des ventes totales')
print()
print('PRODUITS :')
print(f'  Catégorie leader         : {top_category}')
print(f'  Produit le plus rentable : {top_product}')
print()
print('STRATÉGIE DE REMISE :')
print(f'  % de ventes à perte avec remise > 20% : {loss_rate:.1f}%')
print(f'  Recommandation : limiter les remises à 20% maximum')

## Conclusions

1. **Tendance des ventes** : Les ventes ont augmenté progressivement d'année en année, avec une saisonnalité visible en fin d'année (Q4). La catégorie **Technology** montre la croissance la plus régulière.

2. **Concentration géographique** : Les 5 premiers états représentent une part importante du chiffre d'affaires. La **Californie** et **New York** dominent nettement, ce qui indique une concentration géographique des ventes.

3. **Produits rentables** : Les produits **Technologie** génèrent les marges les plus élevées. Les meilleures opportunités de croissance se trouvent dans cette catégorie.

4. **Problème des remises** : Les remises supérieures à **20%** sont fortement corrélées à des pertes, particulièrement dans la catégorie **Furniture**. Une politique stricte de plafonnement des remises à 20% améliorerait significativement la rentabilité.

5. **Recommandation stratégique** : Concentrer les efforts commerciaux sur les états à fort potentiel non encore exploités, réduire les remises excessives, et prioriser les produits technologiques à haute marge.